# labeled GOES/ABI image chips

In [19]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import xarray as xr

SOURCE = Path("../step2_judge_for_goes/labeled_goes_data.nc")
raw = xr.open_dataset(SOURCE).load()

OUTPUT_ZARR = Path("output/earthcare-marine-stratocumulus-transitions/labeled_goes_data/earthcode_labeled_goes_data.zarr")
OUTPUT_ZARR.parent.mkdir(parents=True, exist_ok=True)

raw

<xarray.Dataset> Size: 251MB
Dimensions:     (sample: 479, channel: 4, y: 128, x: 128)
Coordinates:
  * sample      (sample) int32 2kB 1 2 6 7 0 1 4 6 0 2 5 ... 6 7 8 9 0 2 4 7 8 9
Dimensions without coordinates: channel, y, x
Data variables:
    unique_id   (sample) object 4kB '202501021600_2304_128' ... '202508201400...
    class_name  (sample) object 4kB 'Clustered Cu' ... 'Closed MCC'
    image_data  (sample, channel, y, x) float64 251MB 0.469 0.5521 ... 0.3779
    datetime    (sample) datetime64[ns] 4kB 2025-01-02T16:00:00 ... 2025-08-2...
    y_idx       (sample) float64 4kB 2.304e+03 1.792e+03 ... 896.0 1.152e+03
    x_idx       (sample) float64 4kB 128.0 1.088e+03 1.792e+03 ... 768.0 832.0
    lat_mid     (sample) float64 4kB -24.31 -19.1 -21.73 ... -10.55 -12.95
    lon_mid     (sample) float64 4kB -94.53 -85.06 -77.94 ... -88.03 -87.55
    ice_flag    (sample) float64 4kB 0.01538 0.005859 0.0 0.0 ... 0.0 0.0 0.0
    label       (sample) float64 4kB 4.0 4.0 1.0 4.0 3.0 ... 1.0 1.0 4.0 1.0 1.0

## Labels and coordinates


In [20]:
CLASS_NAMES = np.array(['Solid stratus', 'Closed MCC', 'Open MCC', 'Disorganized MCC', 'Clustered Cu', 'Suppressed Cu'])

In [21]:
ds = raw.drop_vars("class_name").assign_coords(
    sample=np.arange(raw.sizes["sample"], dtype="int32"),
    channel=['BT15_minus_BT14', 'BT14_minus_BT11', 'BT14', 'COD'],
    y=np.arange(raw.sizes["y"], dtype="int16"),
    x=np.arange(raw.sizes["x"], dtype="int16"),
)
ds["source_sample_index"] = ("sample", raw.sample.values.astype("int16"))
ds["label"] = ("sample", raw.label.values.astype("int16"))
ds["label_name"] = ("sample", CLASS_NAMES[ds.label.values])
ds["unique_id"] = ("sample", raw.unique_id.values.astype(str))
ds = ds.set_coords(["datetime", "lat_mid", "lon_mid"])
ds


<xarray.Dataset> Size: 251MB
Dimensions:              (sample: 479, channel: 4, y: 128, x: 128)
Coordinates:
  * sample               (sample) int32 2kB 0 1 2 3 4 5 ... 474 475 476 477 478
  * channel              (channel) <U15 240B 'BT15_minus_BT14' ... 'COD'
  * y                    (y) int16 256B 0 1 2 3 4 5 ... 122 123 124 125 126 127
  * x                    (x) int16 256B 0 1 2 3 4 5 ... 122 123 124 125 126 127
    datetime             (sample) datetime64[ns] 4kB 2025-01-02T16:00:00 ... ...
    lat_mid              (sample) float64 4kB -24.31 -19.1 ... -10.55 -12.95
    lon_mid              (sample) float64 4kB -94.53 -85.06 ... -88.03 -87.55
Data variables:
    unique_id            (sample) <U22 42kB '202501021600_2304_128' ... '2025...
    image_data           (sample, channel, y, x) float64 251MB 0.469 ... 0.3779
    y_idx                (sample) float64 4kB 2.304e+03 1.792e+03 ... 1.152e+03
    x_idx                (sample) float64 4kB 128.0 1.088e+03 ... 768.0 832.0
    ice_flag             (sample) float64 4kB 0.01538 0.005859 0.0 ... 0.0 0.0
    label                (sample) int16 958B 4 4 1 4 3 4 4 3 ... 1 3 1 1 1 4 1 1
    source_sample_index  (sample) int16 958B 1 2 6 7 0 1 4 6 ... 8 9 0 2 4 7 8 9
    label_name           (sample) <U16 31kB 'Clustered Cu' ... 'Closed MCC'

## Metadata


In [22]:
ds.attrs['title'] = 'GOES ABI labeled image chips for mesoscale cloud morphology classification'
ds.attrs['summary'] = 'Manual labels and normalized four-channel GOES image chips for cloud morphology classification.'
ds.attrs['keywords'] = 'GOES, ABI, stratocumulus, cloud morphology, closed cells, open cells, CNN, training data'
ds.attrs['source_processing_script'] = 'step2_judge_for_goes/judge_goes.py'
ds.attrs['license'] = 'CC-BY-4.0'
ds.attrs['spatial_resolution'] = 'Approximately 1 km after 2x2 interpolation (from original 2km nadir resolution) in preprocessing workflow; each chip is 128 x 128 local pixels.'
ds.attrs['platform'] = 'GOES-16 (until 7 Apr. 2025), and GOES-19 (from 8 Apr. 2025)'
ds.attrs['instrument'] ='GOES-East Advanced Baseline Imager (ABI)'

In [23]:
# Added chip and label metadata.
ds['image_data'].attrs["long_name"] = 'normalized four-channel GOES image chip'
ds['image_data'].attrs["units"] = '1'
ds['image_data'].attrs["grid_mapping"] = 'spatial_ref'
ds['image_data'].attrs["ancillary_variables"] = 'label label_name'
ds['image_data'].attrs["comment"] = 'Channels are BT15-BT14, BT14-BT11, BT14, and normalized cloud optical depth. Values are normalized for CNN input.'
ds['channel'].attrs["long_name"] = 'model input channel'
ds['channel'].attrs['description'] = "Derived GOES channels used by Johanna's CNN preprocessing: BT15_minus_BT14, BT14_minus_BT11, BT14, and cloud optical depth (per chip normalized)."
ds['channel'].attrs['source_variables'] = "BT11, BT14 and BT15 are brightness temperatures computed from IR radiances of ABI channels 11, 14, and 15, from the ABI-L1b-RadF product; cloud optical depth is from the ABI-L2-COD2KMF product."
ds['label'].attrs["long_name"] = 'manual mesoscale cloud morphology class'
ds['label'].attrs["units"] = '1'
ds['label'].attrs["flag_values"] = np.arange(len(CLASS_NAMES), dtype='int16')
ds['label'].attrs["flag_meanings"] = ' '.join((name.lower().replace(' ', '_') for name in CLASS_NAMES))
ds['label_name'].attrs["long_name"] = 'human-readable class name derived from label'
ds['unique_id'].attrs["long_name"] = 'stable GOES chip identifier'
ds['unique_id'].attrs["comment"] = 'Format is YYYYMMDDHHMM_yindex_xindex.'
ds['datetime'].attrs["long_name"] = 'GOES scene datetime'
ds['datetime'].attrs["standard_name"] = 'time'
ds['ice_flag'].attrs["long_name"] = 'fraction of chip pixels colder than 273 K'
ds['ice_flag'].attrs["units"] = '1'
ds['ice_flag'].attrs["valid_min"] = 0.0
ds['ice_flag'].attrs["valid_max"] = 1.0
ds['source_sample_index'].attrs["long_name"] = 'original non-unique sample coordinate from source file'
ds['source_sample_index'].attrs["units"] = '1'
ds['sample'].attrs["long_name"] = 'unique image chip sample index'
ds['sample'].attrs["units"] = '1'

for axis, coordinate, units in [("lat", "latitude", "degrees_north"), ("lon", "longitude", "degrees_east")]:
    ds[f"{axis}_mid"].attrs["long_name"] = f"chip center {coordinate}"
    ds[f"{axis}_mid"].attrs["standard_name"] = coordinate
    ds[f"{axis}_mid"].attrs["units"] = units
for axis, label in {"x": "column", "y": "row"}.items():
    ds[axis].attrs["long_name"] = f"chip-local image {label} pixel index"
    ds[axis].attrs["units"] = "1"
    ds[axis].attrs["comment"] = f"Local {axis} coordinate within the 128-pixel chip."
    ds[f"{axis}_idx"].attrs["long_name"] = f"GOES processing-grid {axis} index at chip center"
    ds[f"{axis}_idx"].attrs["units"] = "1"
    ds[f"{axis}_idx"].attrs["comment"] = "Chip-center index in the GOES processing grid used by the labeling workflow."


## Save Zarr


In [24]:
ds.attrs["date_created"] = datetime.now(timezone.utc).isoformat()
for variable in ds.variables.values():
    variable.encoding["chunks"] = variable.shape

ds.to_zarr(OUTPUT_ZARR, mode="w", consolidated=True, zarr_format=2)
OUTPUT_ZARR


PosixPath('output/earthcare-marine-stratocumulus-transitions/labeled_goes_data/earthcode_labeled_goes_data.zarr')

## Check source values

In [25]:
# Skip class_name (replaced by label_name) and the added sample/channel/x/y coordinates.
# The original sample values are checked through source_sample_index.
original = xr.open_dataset(SOURCE)
saved = xr.open_zarr(OUTPUT_ZARR, consolidated=True)
for name in original.drop_vars("class_name").variables:
    np.testing.assert_array_equal(
        saved["source_sample_index" if name == "sample" else name].values,
        original[name].values, err_msg=name,
    )
np.testing.assert_array_equal(saved.label_name.values, CLASS_NAMES[original.label.values.astype(int)])
